# R14-H190 + R14-H153 - the glyph operator and the header co-processor

**Author**: Knowledge Graph Foundry autonomous build (kj) <br>
**Date**: 2026-07-07 <br>
**Pipeline stage**: R14 ingest-fidelity round - deterministic, CPU-only, no LLM/GPU <br>
**Graph**: rebuilt CPAP corpus (neo4j2, read-only) - entity names + provenance <br>

Two deterministic ingest-fidelity operators, measured against pre-registered bars.

**H190 - glyph normalization operator**: H147 dissolved the all-parser-absent mode family into a
trademark/glyph matching artifact - symbol-stripped matching alone rescues 69/676 of the H51 absent
set with zero parser change. This builds the general operator (strip trademark glyphs, NFKC ligatures /
fullwidth / non-breaking spaces, unicode punctuation variants) applied at extraction post-process AND
evidence matching, and tests three clauses: (a) recover >= 60 of the 69 end-to-end, (b) >= 10% of the
H107 66-pair variance set becomes exact-match, (c) zero false conflations across the corpus vocabulary.

**H153 - header carryover**: the minimal fix for H144's 64 severed table rows - keep a table atomic
when it fits the chunk budget, re-print the header row + separator at the top of every continuation
chunk when it must split. A chunker post-transform (the shipped module is untouched; the logic is
copied here). Clauses: (a) zero severed rows post-transform, (b) token overhead <= 2%; refuted if
table-boundary detection injects false headers into prose at > 1%.

## Outputs
- `reports/glyph-carryover-r14-<stamp>.json`


In [1]:
# Imports
# stdlib
import os, sys, re, json, unicodedata, collections, datetime
from pathlib import Path
# third party
import tiktoken
from neo4j import GraphDatabase
from rich import print as rprint
from rich.progress import Progress

# resolve project root (notebook runs from notebooks/); make paths cwd-independent
ROOT = Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "notebooks"))
sys.path.insert(0, str(ROOT / "src"))

from parser_harness import load_entities, load_text, norm, nospace, present_in, TRIO
from knowledge_graph_foundry.ingest.readers import read_document
from knowledge_graph_foundry.ingest.chunking import chunk_document, _snap_to_boundary

NEO4J_URI = "bolt://user-konrad.jelen-kgf-neo4j2:7687"   # read-only neo4j2 (NOT the .env live graph)
DOC_DIR = ROOT / "data/external/cpap-datasheets-and-manuals"
CHUNK_SIZE, CHUNK_OVERLAP = 2000, 200
enc = tiktoken.get_encoding("cl100k_base")
stamp = datetime.datetime.now(datetime.timezone.utc).strftime("%Y%m%d-%H%M%S")
rprint(f"[bold]config[/bold] root={ROOT.name}  chunk {CHUNK_SIZE}/{CHUNK_OVERLAP}  stamp {stamp}")


2026-07-07 16:11:57.475 | INFO     | knowledge_graph_foundry.config:<module>:40 - PROJ_ROOT path is: /home/lab/workspace/learning/projects/knowledge-graph-foundry


config root=knowledge-graph-foundry  chunk 2000/200  stamp 20260707-141157

## H190 - the glyph normalization operator

Strip trademark glyphs before NFKC (NFKC maps the U+2122 glyph to the letters `TM`, which would
corrupt the match), then NFKC-fold (ligatures, fullwidth, non-breaking space to space), then translate
unicode punctuation variants (dash family to `-`, curly quotes to straight, exotic spaces to space,
zero-width/soft-hyphen deleted), then lowercase and collapse whitespace. The `nospace` variant drops
all whitespace for the space-insensitive match used by the H51 harness convention.

In [2]:
# The operator - deterministic, applied at BOTH extraction post-process and evidence matching
_SYM_DELETE = dict.fromkeys(map(ord, "™®©℠℗"), None)   # TM R C SM P
_PUNCT = {
    0x2010:"-",0x2011:"-",0x2012:"-",0x2013:"-",0x2014:"-",0x2015:"-",0x2212:"-",
    0x2018:"'",0x2019:"'",0x201a:"'",0x201b:"'",0x201c:'"',0x201d:'"',0x201e:'"',
    0x00a0:" ",0x2007:" ",0x202f:" ",0x2009:" ",0x200a:" ",0x2002:" ",0x2003:" ",
    0x2004:" ",0x2005:" ",0x2006:" ",0x2008:" ",0x3000:" ",
    0x200b:"",0x200c:"",0x200d:"",0xfeff:"",0x00ad:"",
}
def glyph_norm(x):
    x = x or ""
    x = x.translate(_SYM_DELETE)          # strip glyphs BEFORE NFKC
    x = unicodedata.normalize("NFKC", x)  # ligatures, fullwidth, nbsp -> space, superscripts
    x = x.translate(_SYM_DELETE)          # NFKC can re-expose composed glyphs
    x = x.translate(_PUNCT)
    return " ".join(x.lower().split())
def glyph_nospace(x):
    return re.sub(r"\s+", "", glyph_norm(x))
def present_glyph(name, dn, gmaps):
    if dn not in gmaps: return False
    gn, gs = gmaps[dn]
    nn = glyph_norm(name)
    if nn and nn in gn: return True
    ns = glyph_nospace(name)
    return bool(ns) and ns in gs

for s in ["SleepStyle™ Auto", "Ultra‑Fine Filter", "Air Filter"]:
    rprint(f"  [dim]{s!r}[/dim] -> {glyph_norm(s)!r}")


'SleepStyle™ Auto' -> 'sleepstyle auto'

'Ultra‑Fine Filter' -> 'ultra-fine filter'

'Air Filter' -> 'air filter'

In [3]:
# Clause (a): re-parse the H51 loss harness under the operator
docnames, rows = load_entities()
trio = {p: load_text(p) for p in TRIO}
tm = {p: {d:(norm(t),nospace(t)) for d,t in trio[p].items()} for p in TRIO}          # strict H51
gm = {p: {d:(glyph_norm(t),glyph_nospace(t)) for d,t in trio[p].items()} for p in TRIO}  # glyph

def strict_present_any(name, docs):
    return any((d in tm[p]) and present_in(name, tm[p][d][0], tm[p][d][1]) for p in TRIO for d in docs)
def glyph_present_any(name, docs):
    return any(present_glyph(name, d, gm[p]) for p in TRIO for d in docs)

# reproduce the 676 absent + the 69 sym-strip-rescuable (H147 present2: strips U+2122/AE/A9 only)
absent = [r for r in rows if not strict_present_any(r["name"], r["docs"])]
SYM = re.compile("[™®©]")
def _n2(x): return norm(SYM.sub(" ", x or ""))
def _s2(x): return nospace(SYM.sub("", x or ""))
sm2 = {p: {d:(_n2(t),_s2(t)) for d,t in trio[p].items()} for p in TRIO}
def present2(name, dn, mp):
    if dn not in mp: return False
    tn,ts=mp[dn]; nn=_n2(name)
    if nn and nn in tn: return True
    ns=_s2(name); return bool(ns) and ns in ts
rescued69 = [r for r in absent if any(present2(r["name"],d,sm2[p]) for p in TRIO for d in r["docs"])]

recovered = [r for r in rescued69 if glyph_present_any(r["name"], r["docs"])]
not_rec = [r["name"] for r in rescued69 if r not in recovered]
h190a = dict(absent_total=len(absent), sym_strip_rescuable=len(rescued69),
             recovered_glyph=len(recovered), bar=60,
             passed=len(recovered) >= 60, not_recovered=not_rec)
rprint(f"[bold]H190(a)[/bold] absent={len(absent)}  sym-strip-rescuable={len(rescued69)}  "
       f"recovered under operator=[yellow]{len(recovered)}[/yellow]/{len(rescued69)}  bar>=60  "
       f"[{'green' if h190a['passed'] else 'red'}]{'PASS' if h190a['passed'] else 'FAIL'}[/]")


H190(a) absent=676  sym-strip-rescuable=69  recovered under operator=69/69  bar>=60  PASS

In [4]:
# Clause (b): H107 66-pair surface-form variance set -> exact-match under normalization
h107 = json.load(open(ROOT / "reports/identity-forensics-r11-final-20260706-205147.json"))
pairs = h107["pairs"]
exact = [(p["a"],p["b"]) for p in pairs if glyph_norm(p["a"]) == glyph_norm(p["b"])]
exact_ns = [(p["a"],p["b"]) for p in pairs if glyph_nospace(p["a"]) == glyph_nospace(p["b"])]
frac = len(exact)/len(pairs) if pairs else 0.0
h190b = dict(n_pairs=len(pairs), exact_norm=len(exact), exact_nospace=len(exact_ns),
             frac=frac, bar=0.10, passed=frac >= 0.10, exact_pairs=exact)
rprint(f"[bold]H190(b)[/bold] pairs={len(pairs)}  exact-match under operator=[yellow]{len(exact)}[/yellow] "
       f"({frac*100:.1f}%)  bar>=10%  [{'green' if h190b['passed'] else 'red'}]"
       f"{'PASS' if h190b['passed'] else 'FAIL'}[/]")
rprint("[dim]the 66 pairs differ by tokens/word-order/expansions, not glyphs - normalization is orthogonal[/dim]")


H190(b) pairs=66  exact-match under operator=0 (0.0%)  bar>=10%  FAIL

the 66 pairs differ by tokens/word-order/expansions, not glyphs - normalization is orthogonal

In [5]:
# Clause (c): full corpus name vocabulary from neo4j2 (read-only) -> pairwise collision of normalized forms
drv = GraphDatabase.driver(NEO4J_URI, auth=("neo4j","kgfoundry"), notifications_min_severity="OFF")
with drv.session() as s:
    vocab = sorted({r["name"] for r in s.run("MATCH (e:Entity) WHERE e.name IS NOT NULL RETURN e.name AS name")
                    if r["name"]})
drv.close()

groups = collections.defaultdict(list)
for nm in vocab:
    groups[glyph_norm(nm)].append(nm)
collisions = {k:v for k,v in groups.items() if len(v) > 1}
def only_case_glyph_variant(names):
    keys = {re.sub(r"[^a-z0-9]", "", n.lower()) for n in names}
    return len(keys) == 1
false_conflations = {k:v for k,v in collisions.items() if not only_case_glyph_variant(v)}
h190c = dict(vocab=len(vocab), collision_groups=len(collisions),
             collisions=[{"form":k,"names":v,"case_glyph_variant_only":only_case_glyph_variant(v)}
                         for k,v in collisions.items()],
             false_conflations=len(false_conflations), bar=0, passed=len(false_conflations) == 0)
rprint(f"[bold]H190(c)[/bold] vocab={len(vocab)}  collision groups={len(collisions)}  "
       f"false conflations=[yellow]{len(false_conflations)}[/yellow]  bar=0  "
       f"[{'green' if h190c['passed'] else 'red'}]{'PASS' if h190c['passed'] else 'FAIL'}[/]")
for k,v in collisions.items():
    kind = "case/glyph variant (SAFE)" if only_case_glyph_variant(v) else "DISTINCT NAMES (FALSE CONFLATION)"
    rprint(f"  [dim]{k!r}[/dim] <- {[x for x in v]}  [{kind}]")


H190(c) vocab=2797  collision groups=1  false conflations=0  bar=0  PASS

'air filter cover' <- ['Air Filter Cover', 'Air filter cover']

## H153 - header carryover (the minimal co-processor)

The shipped `chunk_document` (token window, sentence snap) is untouched. Its logic is copied here with
char-offset instrumentation (parity-checked against the shipped chunker), then a post-transform:
for every markdown table (header row + `---` separator), any chunk that holds a data row of that table
but not its header gets the header row and separator re-printed at the top. Tables that fit a single
chunk are already atomic and untouched. The H144 severance audit is re-run on the transformed chunk
texts; token overhead and false-header injection are measured over the full corpus chunk stream.

In [6]:
# Copied chunker logic (shipped module untouched) + char-offset spans + H144 table detection
def chunks_with_spans(doc):
    text = doc.text
    if not text.strip(): return []
    tokens = enc.encode(text); total = len(tokens)
    if total <= CHUNK_SIZE:
        return [{"index":0,"text":text,"cstart":0,"cend":len(text)}]
    out=[]; start=index=0
    while start < total:
        end = min(start+CHUNK_SIZE, total)
        piece = enc.decode(tokens[start:end])
        if end < total: piece = _snap_to_boundary(piece, enc, CHUNK_SIZE)
        if not piece.strip(): start = end; continue
        tc = len(enc.encode(piece))
        cstart = len(enc.decode(tokens[:start]))
        out.append({"index":index,"text":piece,"cstart":cstart,"cend":cstart+len(piece)})
        index += 1
        if end >= total: break
        start += max(tc-CHUNK_OVERLAP, 1)
    return out

SEP_RE = re.compile(r"^\s*\|?\s*:?-{2,}:?\s*(\|\s*:?-{2,}:?\s*)+\|?\s*$")
ROW_RE = re.compile(r"^\s*\|.*\|\s*$")
def line_spans(text):
    out=[]; pos=0
    for ln in text.splitlines(keepends=True):
        out.append((ln.rstrip("\n"), pos, pos+len(ln.rstrip("\n")))); pos += len(ln)
    return out
def tables(text):
    ls=line_spans(text); out=[]; i=0
    while i < len(ls)-1:
        htxt,ha,hb = ls[i]; stxt,_,_ = ls[i+1]
        if ROW_RE.match(htxt) and SEP_RE.match(stxt):
            rows=[]; j=i+2
            while j < len(ls) and ROW_RE.match(ls[j][0]):
                rows.append((ls[j][1], ls[j][2])); j += 1
            out.append({"header":(ha,hb),"htxt":htxt,"stxt":stxt,"ncols":htxt.count("|"),"rows":rows})
            i = j
        else: i += 1
    return out
def chunk_of(span, ch):
    a,b = span; return [c["index"] for c in ch if c["cstart"] <= a and b <= c["cend"]]

docs=[]
pdfs = sorted(DOC_DIR.glob("*.pdf"))
with Progress() as pr:
    t = pr.add_task("parse+chunk", total=len(pdfs))
    for p in pdfs:
        try: d = read_document(p)
        except Exception as e:
            rprint(f"[red]parse fail[/red] {p.name}: {e}"); pr.advance(t); continue
        ch = chunks_with_spans(d)
        parity = [c["text"] for c in ch] == [c.text for c in chunk_document(d, CHUNK_SIZE, CHUNK_OVERLAP)]
        docs.append({"name":p.name,"text":d.text,"chunks":ch,"parity":parity})
        pr.advance(t)
rprint(f"parsed [yellow]{len(docs)}[/yellow] docs  chunk-parity "
       f"[green]{sum(d['parity'] for d in docs)}/{len(docs)}[/green]")


/home/lab/workspace/learning/projects/knowledge-graph-foundry/.venv/lib/python3.12/site-packages/rich/live.py:260: 
UserWarning: install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

2026-07-07 16:12:00.791 | DEBUG    | knowledge_graph_foundry.ingest.readers:read_document:68 - read 
0-20190113114505.pdf (4881 chars)

2026-07-07 16:12:05.213 | DEBUG    | knowledge_graph_foundry.ingest.readers:read_document:68 - read 
1017900r4_ResMed_Product_Catalogue_ANZ_Eng_LowRes.pdf (14762 chars)

2026-07-07 16:12:05.222 | DEBUG    | knowledge_graph_foundry.ingest.chunking:chunk_document:65 - chunked 
d_c544d2031f25c004 into 2 chunks

2026-07-07 16:12:16.244 | DEBUG    | knowledge_graph_foundry.ingest.readers:read_document:68 - read 
3B_User-Manual_CPAP-Auto-CPAP_RESmart_BMC_V1.7_ENG-1.pdf (52638 chars)

...

2026-07-07 16:14:55.198 | DEBUG    | knowledge_graph_foundry.ingest.readers:read_document:68 - read 
moh-adp-product-manual-respiratory-devices-airway-clearance-en-2023-06-14.pdf (7326 chars)

2026-07-07 16:14:55.206 | DEBUG    | knowledge_graph_foundry.ingest.chunking:chunk_document:65 - chunked 
d_160ebdb9b2e36f50 into 2 chunks

2026-07-07 16:15:46.357 | DEBUG    | knowledge_graph_foundry.ingest.readers:read_document:68 - read 
product_and_solutions_catalog.pdf (210634 chars)

2026-07-07 16:15:46.569 | DEBUG    | knowledge_graph_foundry.ingest.chunking:chunk_document:65 - chunked 
d_2b3d647422cbe53e into 29 chunks

parsed 28 docs  chunk-parity 28/28

In [7]:
# Baseline severance (reproduce H144 = 64) + transform + post-audit + overhead + injection
def severance_charspan(doc):
    ch=doc["chunks"]; sev=0
    if len(ch) < 2: return 0
    for t in tables(doc["text"]):
        hch=set(chunk_of(t["header"], ch))
        for r in t["rows"]:
            rch=set(chunk_of(r, ch))
            if rch and not (rch & hch): sev += 1
    return sev

base_sev = sum(severance_charspan(d) for d in docs)
base_tokens = sum(len(enc.encode(c["text"])) for d in docs for c in d["chunks"])
total_chunks = sum(len(d["chunks"]) for d in docs)

inj_events = 0; false_inj = 0
def transform(doc):
    global inj_events, false_inj
    ch=[dict(c) for c in doc["chunks"]]
    if len(ch) < 2: return ch
    prepend={ci:[] for ci in range(len(ch))}
    gflags={ci:[] for ci in range(len(ch))}
    for t in tables(doc["text"]):
        hch=set(chunk_of(t["header"], ch))
        hdr_block = t["htxt"] + "\n" + t["stxt"]
        genuine = t["ncols"] >= 2 and bool(SEP_RE.match(t["stxt"])) and len(t["rows"]) >= 1
        for r in t["rows"]:
            for ci in (set(chunk_of(r, ch)) - hch):
                if hdr_block not in prepend[ci] and hdr_block not in ch[ci]["text"]:
                    prepend[ci].append(hdr_block); gflags[ci].append(genuine)
    for ci in range(len(ch)):
        for k, hdr_block in enumerate(prepend[ci]):
            ch[ci]["text"] = hdr_block + "\n" + ch[ci]["text"]
            inj_events += 1
            if not gflags[ci][k]: false_inj += 1
    return ch

def severance_text(doc, tch):
    sev=0
    if len(tch) < 2: return 0
    for t in tables(doc["text"]):
        hdr = t["htxt"]
        for a,b in t["rows"]:
            rt = doc["text"][a:b]
            holders = [c for c in tch if rt in c["text"]]
            if not holders: continue
            if not any(hdr in c["text"] for c in holders): sev += 1
    return sev

post_sev = 0; post_tokens = 0
for d in docs:
    tch = transform(d)
    post_sev += severance_text(d, tch)
    post_tokens += sum(len(enc.encode(c["text"])) for c in tch)

overhead = (post_tokens - base_tokens) / base_tokens if base_tokens else 0.0
inj_rate = false_inj / total_chunks if total_chunks else 0.0
h153 = dict(baseline_severed=base_sev, post_severed=post_sev,
            base_tokens=base_tokens, post_tokens=post_tokens, token_overhead=overhead,
            header_injections=inj_events, false_injections=false_inj, injection_rate=inj_rate,
            total_chunks=total_chunks, bar_severed=0, bar_overhead=0.02, bar_injection=0.01,
            passed_severance=post_sev == 0, passed_overhead=overhead <= 0.02, passed_injection=inj_rate <= 0.01)
rprint(f"[bold]H153[/bold] baseline severed=[yellow]{base_sev}[/yellow] (H144=64)  "
       f"post-transform severed=[yellow]{post_sev}[/yellow] (bar 0)")
rprint(f"       token overhead=[yellow]{overhead*100:.3f}%[/yellow] (bar<=2%)  "
       f"header injections={inj_events}  false injections={false_inj}  "
       f"injection rate=[yellow]{inj_rate*100:.4f}%[/yellow] (bar<1%)")
rprint(f"       [{'green' if (h153['passed_severance'] and h153['passed_overhead'] and h153['passed_injection']) else 'red'}]"
       f"severance {'PASS' if h153['passed_severance'] else 'FAIL'} / "
       f"overhead {'PASS' if h153['passed_overhead'] else 'FAIL'} / "
       f"injection {'PASS' if h153['passed_injection'] else 'FAIL'}[/]")


H153 baseline severed=64 (H144=64)  post-transform severed=0 (bar 0)

token overhead=0.258% (bar<=2%)  header injections=18  false injections=0  injection rate=0.0000% (bar<1%)

severance PASS / overhead PASS / injection PASS

## Report

In [8]:
report = {
    "batch": "R14-H190 + R14-H153",
    "generated_utc": stamp,
    "environment": "CPU-only, deterministic, no LLM/GPU; parser=pymupdf4llm (current project reader)",
    "H190": {
        "operator": "strip trademark glyphs -> NFKC -> punctuation-variant translate -> lower+collapse",
        "clause_a_loss_recovery": h190a,
        "clause_b_h107_exact_match": h190b,
        "clause_c_vocab_collision": h190c,
        "verdict": ("PARTIAL: (a) PASS, (c) PASS (no false conflation), (b) FAIL - "
                    "H107 variance is token/word-level, not glyph-level, so normalization makes 0 pairs exact"),
    },
    "H153": {
        "note": ("Docling swap (H146) would change the input text but the header-carryover logic is "
                 "format-generic; measured here on the current pymupdf4llm output to isolate the chunker fix"),
        **h153,
        "verdict": "CONFIRMED if post_severed==0 and overhead<=2% and injection<=1%",
    },
}
out = ROOT / f"reports/glyph-carryover-r14-{stamp}.json"
out.write_text(json.dumps(report, indent=2, default=str))
rprint(f"[green]saved[/green] {out}")


saved 
/home/lab/workspace/learning/projects/knowledge-graph-foundry/reports/glyph-carryover-r14-20260707-141157.json